# eph_07 — Bout encoding

Do LC units respond differently to tongue-movement **bouts** occurring within a trial
(go-responsive) vs. during the **inter-trial interval** (ITI)? This is the ephys
counterpart to the covert-preparatory-movement story `kin_02` and `kin_05` tell on the
behavior side: bouts of movement happen outside of licked, cue-triggered responses, and
this notebook asks whether LC units treat them the same way.

**Pipeline:**
1. Bout segmentation — `annotate_movement_bouts` + `classify_bout_times` /
   `get_session_bout_times` (`ephys_utils.py`): group movements into bouts by
   inter-movement gap, then split bout onsets into within-trial vs ITI by their timing
   relative to go cues. **Locally testable** — exercised directly below against the
   pooled parquet.
2. Data loading via `data_loading.py` (Code Ocean only — per-session movements, trials,
   licks, and spike times).
3. Example single-unit raster + PSTH aligned to bout starts, reusing `eph_00`'s
   `make_rp_and_events` / `compute_psth` / `smooth_vector` / `plot_psth` path.
4. Population PETH across all units — heatmap, mean±SEM overlay, waterfall — read
   against a go-cue-aligned reference.
5. Per-unit encoding: paired go-responsive vs ITI Δ firing rate (Wilcoxon), registered
   through `per_unit_stats_registry` / `encoding_methods` so it composes with
   `eph_01`–`eph_04`.
6. Qualitative single-trial event raster and timeseries.
7. Robustness check: the same comparison using lick-bout onsets instead of
   movement-bout onsets.

Ported from `tongue_kinematics_ephys_intertrialmovs.ipynb` (cells 10–24 for the
movement-bout primary path, 29–31 for the per-unit encoding comparison, 35–36 for the
qualitative figures; cell 37's video-clip extraction is dropped — a one-off, and the
clip helpers are library-owned). See `TODO.md`'s `eph_07_bout_encoding` item and the
"Port plan for the five HOLD notebooks" section above it.

**What actually ran this session:** only §4 (the bout-helper verification), against
`data/for_local/all_tongue_movements_04022026.parquet`. Everything from §5 onward needs
per-session spike times and intermediate data that only exist on Code Ocean — written
and reasoned through, but **not executed**. Each such section says so again at the point
it stops being testable.


## 1. Setup

In [1]:
%matplotlib inline

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, spearmanr

from plotstyle import apply_style, PALETTE, OKABE_ITO, style_ax, save_fig
apply_style()


In [2]:
# ── Environment detection ─────────────────────────────────────────────────
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "eph_07_bout_encoding"
SAVE_FIG = False

print(f"ENV       : {ENV}")
print(f"FOR_LOCAL : {FOR_LOCAL}")


ENV       : local
FOR_LOCAL : /Users/mib/Documents/Code/kinematics_analysis/data/for_local


## 2. Load data

Full pipeline (combined unit table → QC/session filter → per-unit spike times) is
Code Ocean only. Locally, the only thing we can load is the pooled kinematics parquet,
used below to exercise the bout helpers directly (§4) — it carries `start_time`,
`trial`, `session`, and `goCue_start_time_in_session`, which is everything
`annotate_movement_bouts` and `get_session_bout_times` need.


In [3]:
from data_loading import (
    load_session_quality_filter,
    filter_ephys_units,
    load_units_with_spike_times,
)

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)

    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)

    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    # Local dev: no spike times available. Load the pooled kinematics parquet for the
    # bout-helper verification in §4; everything from §5 on needs spike times and is
    # guarded to not execute here.
    units_with_spikes = None
    base_dirs = [FOR_LOCAL]
    movements_local = pd.read_parquet(FOR_LOCAL / "all_tongue_movements_04022026.parquet")
    print(f"Local dev: movements_local {movements_local.shape}, "
          f"{movements_local['session'].nunique()} sessions")


Local dev: movements_local (246359, 49), 44 sessions


## 3. Bout segmentation: definitions and thresholds

Two ways to define a "bout" of movement, per the source notebook:

1. **Movement-derived (primary).** `annotate_movement_bouts` groups raw tongue
   movements by inter-movement gap (`gap_threshold_s`); a bout's onset is its first
   movement's `start_time`. This is the definition used for every section below except
   §9. It doesn't inherit the lickometer's blind spot for non-lick movements — the
   entire premise `kin_05` establishes — so it's the better primary.
2. **Lick-derived (robustness check, §9).** Bout onsets come from
   `licks["bout_start"]`, produced upstream by
   `aind_dynamic_foraging_basic_analysis.licks.lick_analysis`. Kept at lower weight —
   one comparison, not a fully duplicated analysis.

Both definitions then classify bout onsets as **go-responsive** (within-trial) or
**ITI** relative to the session's go cues, via the shared `classify_bout_times`.

The source notebook drifted on thresholds across cells — `GO_RESPONSE_WINDOW_S` /
`ITI_MIN_POST_CUE_S` / `ITI_MIN_PRE_NEXT_S` were `2.0/2.0/1.0` at cell 12 and every
population-PETH cell (14/15/19/22), but `1.0/2.0/0.5` at cell 29 (the per-unit
encoding comparison) and `2.0/1.5/1.0` at cell 26 (the lick-bout section's own
header cell). We pin **one** set for the movement-derived primary path — the
`2.0/2.0/1.0` set, since it's what 5 of the 6 movement-derived cells actually used —
and use it everywhere in §5–§8. §9's lick-derived robustness check keeps its own
thresholds (`2.0/1.5/1.0`, from source cell 26) rather than forcing a single global
set onto a genuinely different event stream; both sets are stated explicitly at their
point of use so this doesn't read as an unexplained discrepancy.


In [4]:
from ephys_utils import annotate_movement_bouts, classify_bout_times, get_session_bout_times

# ---- primary (movement-derived) thresholds, used in §5-§8 ----
GAP_THRESHOLD_S      = 0.5   # annotate_movement_bouts: max inter-movement gap within a bout
GO_RESPONSE_WINDOW_S = 2.0   # go-responsive: bout starts within this long after the preceding cue
ITI_MIN_POST_CUE_S   = 2.0   # ITI: bout starts more than this long after the preceding cue
ITI_MIN_PRE_NEXT_S   = 1.0   # ITI: bout starts more than this long before the next cue


## 4. Verify bout helpers locally

The one part of this notebook that can actually run here. Exercises
`annotate_movement_bouts` and `get_session_bout_times` per session against the pooled
parquet, with the pinned primary thresholds from §3.


In [5]:
if ENV == "local":
    movs_valid = movements_local.dropna(subset=["start_time"]).copy()

    # ---- bout annotation across the pool (bout ids are per-session) ----
    movs_annot = annotate_movement_bouts(movs_valid, gap_threshold_s=GAP_THRESHOLD_S)
    n_bouts = movs_annot.groupby("session")["mov_bout_id"].nunique().sum()
    print(f"{len(movs_annot)} movements -> {n_bouts} bouts "
          f"(gap_threshold_s={GAP_THRESHOLD_S}, {movs_annot['session'].nunique()} sessions)")
    print()
    print("Bout size distribution (row-weighted, i.e. one row per movement):")
    print(movs_annot["mov_bout_size"].describe())
    n_singletons = int((movs_annot["mov_bout_size"] == 1).sum())
    print(f"Movements that are singleton bouts: {n_singletons} "
          f"({n_singletons / len(movs_annot):.1%})")

    # ---- per-session bout-level size distribution (one row per bout, not per movement) ----
    bout_sizes = (movs_annot.loc[movs_annot["mov_bout_start"]]
                  .groupby("session")["mov_bout_size"])
    print(f"\nBout-level size distribution (n={n_bouts} bouts): "
          f"median={movs_annot.loc[movs_annot['mov_bout_start'], 'mov_bout_size'].median():.0f}, "
          f"mean={movs_annot.loc[movs_annot['mov_bout_start'], 'mov_bout_size'].mean():.2f}")

    # ---- within-trial vs ITI split, per session ----
    # No standalone trials table locally; each movement already carries its trial's
    # goCue_start_time_in_session, so the unique (trial, goCue) pairs per session are
    # an equivalent stand-in for a trials table's one column that get_session_bout_times needs.
    rows = []
    for sess, g in movs_valid.groupby("session"):
        trials_proxy = g[["trial", "goCue_start_time_in_session"]].dropna().drop_duplicates()
        gr_t, it_t = get_session_bout_times(
            g, trials_proxy,
            gap_threshold_s=GAP_THRESHOLD_S,
            go_response_window_s=GO_RESPONSE_WINDOW_S,
            iti_min_post_cue_s=ITI_MIN_POST_CUE_S,
            iti_min_pre_next_s=ITI_MIN_PRE_NEXT_S,
        )
        rows.append({"session": sess, "n_go_responsive": len(gr_t), "n_iti": len(it_t)})

    split_df = pd.DataFrame(rows)
    print("\nWithin-trial (go-responsive) vs ITI bout counts per session:")
    print(split_df.describe())
    print(f"\nTotals: go_responsive={split_df['n_go_responsive'].sum()}, "
          f"iti={split_df['n_iti'].sum()}")
    display(split_df)
else:
    print("ENV == 'codeocean': bout-helper verification already covered locally; "
          "the same helpers are exercised per-session inside the §6 population loop.")


246359 movements -> 31081 bouts (gap_threshold_s=0.5, 44 sessions)

Bout size distribution (row-weighted, i.e. one row per movement):
count    246359.000000
mean         17.879002
std          18.821531
min           1.000000
25%           8.000000
50%          14.000000
75%          21.000000
max         216.000000
Name: mov_bout_size, dtype: float64
Movements that are singleton bouts: 6024 (2.4%)

Bout-level size distribution (n=31081 bouts): median=5, mean=7.93



Within-trial (go-responsive) vs ITI bout counts per session:
       n_go_responsive       n_iti
count        44.000000   44.000000
mean        326.545455  211.295455
std         172.727700  203.292158
min          33.000000   15.000000
25%         158.750000   51.750000
50%         336.000000  157.500000
75%         460.000000  311.000000
max         623.000000  912.000000

Totals: go_responsive=14368, iti=9297


,session,n_go_responsive,n_iti
0,behavior_716325_2024-05-31_10-31-14,600,912
1,behavior_751004_2024-12-20_13-26-07,517,189
2,behavior_751004_2024-12-21_13-28-24,595,266
3,behavior_751004_2024-12-22_13-09-11,598,383
4,behavior_751004_2024-12-23_14-19-57,503,424
5,behavior_751181_2025-02-25_12-12-30,315,111
6,behavior_751181_2025-02-27_11-24-44,161,64
7,behavior_751766_2025-02-11_11-53-32,459,155
8,behavior_751766_2025-02-13_11-31-21,536,302
9,behavior_751766_2025-02-14_11-37-11,601,261


## 5. Example unit: bout-aligned raster + PSTH

**Code Ocean only — not executed this session.** Loads one example session/unit, builds
the movement-bout classification for that session, then reuses `make_rp_and_events` +
`compute_psth` / `smooth_vector` / `plot_psth` (the same path `eph_00` uses) to build a
raster+PSTH pair for each bout condition, treating each bout onset as its own synthetic
"trial" aligned at t=0 — mirrors source cell 12, but built on `get_session_bout_times`
instead of the four duplicated inline copies.


In [6]:
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import (
    find_session_dir, load_intermediate_data, make_rp_and_events,
    compute_psth, smooth_vector, plot_psth,
)

PRE_S, POST_S = 1.0, 2.0
BIN_SIZE      = 0.001
SMOOTH_SIGMA  = 0.025


def load_example_session_and_unit(units_with_spikes, base_dirs, idx=0):
    """
    Convenience loader for exploratory analysis: picks one row from
    units_with_spikes, loads its session's intermediate data, and converts
    spike_times to session time.

    Returns a dict with session, unit_id, spikes_session_time, movs, kins,
    trials, licks, events.
    """
    row = units_with_spikes.iloc[idx]
    session = row.session
    unit_id = row.unit_id

    sdir = find_session_dir(session, roots=base_dirs)
    data = load_intermediate_data(sdir)  # {movs, trials, licks, kins, events}

    session_offset = data["events"].loc[
        data["events"]["event"] == "goCue_start_time", "raw_timestamps"
    ].iloc[0]
    spikes_session_time = np.asarray(row.spike_times, dtype=float) - session_offset

    return {
        "session": session, "unit_id": unit_id,
        "spikes_session_time": spikes_session_time,
        "movs": data["movs"], "kins": data["kins"],
        "trials": data["trials"], "events": data["events"], "licks": data["licks"],
    }


def raster_for_event_times(spikes, event_times, pre, post, bin_size):
    """
    Build a raster + PSTH aligned to arbitrary event times, via make_rp_and_events
    with synthetic "trials" = one per event (each event is its own trial aligned
    at t=0). Reused for the population loop in §6.
    """
    event_times = np.asarray(event_times, dtype=float)
    trials_synth = list(range(len(event_times)))
    event_dicts = {"event": dict(zip(trials_synth, event_times))}
    return make_rp_and_events(
        spikes=spikes, trials=trials_synth, event_dicts=event_dicts,
        events_to_plot=["event"], align_by="event", sort_by="event",
        pre=pre, post=post, bin_size=bin_size,
    )


In [7]:
if ENV == "codeocean":
    idx = units_with_spikes.sample(1).index[0]
    example = load_example_session_and_unit(units_with_spikes, base_dirs, idx=idx)
    session, unit_id, spikes = example["session"], example["unit_id"], example["spikes_session_time"]
    movs, trials = example["movs"], example["trials"]

    gr_t, it_t = get_session_bout_times(
        movs, trials,
        gap_threshold_s=GAP_THRESHOLD_S,
        go_response_window_s=GO_RESPONSE_WINDOW_S,
        iti_min_post_cue_s=ITI_MIN_POST_CUE_S,
        iti_min_pre_next_s=ITI_MIN_PRE_NEXT_S,
    )
    print(f"Session: {session}   Unit: {unit_id}")
    print(f"go-responsive bouts: {len(gr_t)}   ITI bouts: {len(it_t)}")

    rp_gr, _ = raster_for_event_times(spikes, gr_t, PRE_S, POST_S, BIN_SIZE)
    rp_it, _ = raster_for_event_times(spikes, it_t, PRE_S, POST_S, BIN_SIZE)

    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
    for rp, title, ax_r, ax_p in [
        (rp_gr, f"Within-trial bouts (n={len(gr_t)})", axes[0, 0], axes[1, 0]),
        (rp_it, f"ITI bouts (n={len(it_t)})",          axes[0, 1], axes[1, 1]),
    ]:
        rp.plot_raster(ax=ax_r, spike_color="black")
        ax_r.axvline(0, color=PALETTE["neutral"], lw=0.8, alpha=0.7)
        ax_r.set_title(title)
        ax_r.set_ylabel("Bout #")
        style_ax(ax_r)

        psth, _ = compute_psth(rp.raster, bin_size=rp.bin_size)
        psth_sm = smooth_vector(psth, bin_size=rp.bin_size, sigma=SMOOTH_SIGMA)
        plot_psth(rp.bins, psth, psth_sm, ax=ax_p, label="PSTH")
        ax_p.axvline(0, color=PALETTE["neutral"], lw=0.8, alpha=0.7)
        ax_p.set_xlabel("Time from bout start (s)")
        style_ax(ax_p)

    plt.suptitle(f"Unit {int(unit_id)} ({session}) — aligned to movement-bout start", fontsize=13)
    plt.tight_layout()
    save_fig(fig, f"example_unit_{unit_id}_bout_raster_psth", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 6. Population PETH: within-trial vs ITI bout starts

**Code Ocean only — not executed this session.** Consolidates source cells 14, 15, 19,
and 22 — four near-identical copies of the same population-PETH computation — into one
pass. Keeps the **cell 15** normalization (z-score each unit against its own
session-wide firing-rate mean/SD, not a pre-event baseline — the more defensible choice
per `TODO.md`). Order: build the per-unit PETHs and bout-count cache once, check
sampling adequacy (ITI bout count per session, `MIN_BOUTS` filter), *then* plot.

Reuses `raster_for_event_times` + `compute_psth` from §5 (the `eph_00` raster/PSTH
path) rather than the source's hand-rolled `peth_for_unit` — the one place this
notebook doesn't follow that convention is nowhere; a per-unit `RasterPlotter` is
built for every unit × condition here as well, at a coarser 20 ms bin size to keep the
loop over hundreds of units cheap (matching the source's own choice of bin size for
this section, for the same reason).


In [8]:
PRE_POP, POST_POP = 2.0, 2.0
BIN_SIZE_POP       = 0.02   # 20 ms bins — population loop runs over hundreds of units
BASELINE_WIN       = (-1.0, 0.0)

n_bins_pop = int(np.round((PRE_POP + POST_POP) / BIN_SIZE_POP))
bin_centers = np.linspace(-PRE_POP, POST_POP, n_bins_pop, endpoint=False) + BIN_SIZE_POP / 2


def session_wide_stats(spikes, bin_size):
    """Mean and SD of firing rate across the whole session, in uniform bins."""
    spikes = np.sort(spikes)
    if len(spikes) < 2:
        return np.nan, np.nan
    edges = np.arange(spikes[0], spikes[-1] + bin_size, bin_size)
    counts, _ = np.histogram(spikes, bins=edges)
    rates = counts / bin_size
    return rates.mean(), rates.std()


In [9]:
if ENV == "codeocean":
    peth_gr_list, peth_it_list, unit_labels = [], [], []
    mu_list, sd_list = [], []

    session_data_cache = {}
    session_bout_cache = {}

    for u in units_with_spikes.itertuples(index=False):
        sess_u = u.session

        if sess_u not in session_data_cache:
            sdir = find_session_dir(sess_u, roots=base_dirs)
            session_data_cache[sess_u] = load_intermediate_data(sdir)
        if sess_u not in session_bout_cache:
            data = session_data_cache[sess_u]
            session_bout_cache[sess_u] = get_session_bout_times(
                data["movs"], data["trials"],
                gap_threshold_s=GAP_THRESHOLD_S,
                go_response_window_s=GO_RESPONSE_WINDOW_S,
                iti_min_post_cue_s=ITI_MIN_POST_CUE_S,
                iti_min_pre_next_s=ITI_MIN_PRE_NEXT_S,
            )

        gr_t, it_t = session_bout_cache[sess_u]
        if len(gr_t) == 0 or len(it_t) == 0:
            continue

        evnts = session_data_cache[sess_u]["events"]
        offset = evnts.loc[evnts["event"] == "goCue_start_time", "raw_timestamps"].iloc[0]
        spk = np.asarray(u.spike_times, dtype=float) - offset

        rp_gr, _ = raster_for_event_times(spk, gr_t, PRE_POP, POST_POP, BIN_SIZE_POP)
        rp_it, _ = raster_for_event_times(spk, it_t, PRE_POP, POST_POP, BIN_SIZE_POP)
        psth_gr, _ = compute_psth(rp_gr.raster, bin_size=rp_gr.bin_size)
        psth_it, _ = compute_psth(rp_it.raster, bin_size=rp_it.bin_size)

        mu, sd = session_wide_stats(spk, BIN_SIZE_POP)

        peth_gr_list.append(psth_gr)
        peth_it_list.append(psth_it)
        mu_list.append(mu)
        sd_list.append(sd)
        unit_labels.append((sess_u, u.unit_id))

    peth_gr = np.array(peth_gr_list)
    peth_it = np.array(peth_it_list)
    mu_arr  = np.array(mu_list)[:, None]
    sd_arr  = np.where(np.array(sd_list)[:, None] == 0, np.nan, np.array(sd_list)[:, None])

    z_gr = (peth_gr - mu_arr) / sd_arr
    z_it = (peth_it - mu_arr) / sd_arr

    print(f"Population PETH computed for {len(unit_labels)} units "
          f"(units with <1 bout in either condition were skipped)")
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 6b. Sampling adequacy: ITI bout count per session

Cell 18's distribution + cell 19's `MIN_BOUTS` filter, run before building the sorted
population matrices so units from under-sampled sessions don't contaminate the
heatmap/overlay/waterfall below.


In [10]:
if ENV == "codeocean":
    iti_counts = {sess: len(bout_times[1]) for sess, bout_times in session_bout_cache.items()}
    counts = np.array(list(iti_counts.values()))

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.hist(counts, bins=np.arange(0, counts.max() + 5, 20),
            color=PALETTE["neg"], edgecolor="white")
    ax.axvline(np.median(counts), color=PALETTE["neutral"], ls="--", lw=1,
               label=f"median = {int(np.median(counts))}")
    ax.set_xlabel("ITI bouts per session")
    ax.set_ylabel("Sessions")
    ax.set_title(f"ITI bout count distribution (n={len(counts)} sessions)")
    ax.legend()
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "iti_bout_count_per_session", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    print(f"min: {counts.min()}, median: {int(np.median(counts))}, "
          f"max: {counts.max()}, total: {counts.sum()}")

    MIN_BOUTS = 100
    good_sessions = {
        sess for sess, (gr_t, it_t) in session_bout_cache.items()
        if len(gr_t) > MIN_BOUTS and len(it_t) > MIN_BOUTS
    }
    print(f"Sessions with >{MIN_BOUTS} bouts in both: {len(good_sessions)}/{len(session_bout_cache)}")

    session_arr = np.array([lbl[0] for lbl in unit_labels])
    session_mask = np.isin(session_arr, list(good_sessions))
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 6c. Population heatmap

Sorted by signed peak z-score in the post-bout-start window of the within-trial
condition — units with the strongest within-trial response at the top, in both panels.


In [11]:
if ENV == "codeocean":
    post_mask = bin_centers >= 0

    def signed_peak(z, mask):
        seg = z[:, mask]
        pos, neg = np.nanmax(seg, axis=1), np.nanmin(seg, axis=1)
        return np.where(np.abs(pos) >= np.abs(neg), pos, neg)

    peak_gr = signed_peak(z_gr, post_mask)
    valid = (session_mask & np.isfinite(peak_gr)
             & np.all(np.isfinite(z_gr), axis=1) & np.all(np.isfinite(z_it), axis=1))
    order = np.argsort(-peak_gr[valid])
    z_gr_sorted = z_gr[valid][order]
    z_it_sorted = z_it[valid][order]
    n_valid = int(valid.sum())

    print(f"Population heatmap: {n_valid}/{len(unit_labels)} units "
          f"(dropped {len(unit_labels) - n_valid} for missing baseline, NaN PETH, "
          f"or an under-sampled session)")

    vmax = np.nanpercentile(np.abs(np.stack([z_gr_sorted, z_it_sorted])), 98)
    fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
    for ax, Z, title in [(axes[0], z_gr_sorted, "Within-trial bouts"),
                          (axes[1], z_it_sorted, "ITI bouts")]:
        im = ax.imshow(Z, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                        extent=[bin_centers[0], bin_centers[-1], n_valid, 0],
                        interpolation="nearest")
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("Time from bout start (s)")
        ax.set_title(title)
    axes[0].set_ylabel("Unit (sorted by within-trial peak)")
    cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
    cbar.set_label("Firing rate (z, session-wide)")
    plt.suptitle("Population PETH — movement-bout starts", y=1.02)
    save_fig(fig, "population_heatmap_bout_starts", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 6d. Mean ± SEM overlay

In [12]:
if ENV == "codeocean":
    def mean_sem(Z):
        mean = np.nanmean(Z, axis=0)
        n = np.sum(~np.isnan(Z), axis=0)
        sem = np.nanstd(Z, axis=0, ddof=1) / np.sqrt(n)
        return mean, sem

    gr_mean, gr_sem = mean_sem(z_gr_sorted)
    it_mean, it_sem = mean_sem(z_it_sorted)

    fig, ax = plt.subplots(figsize=(6, 4))
    for mean, sem, color, label in [
        (gr_mean, gr_sem, PALETTE["pos"], f"Within-trial (n={n_valid})"),
        (it_mean, it_sem, PALETTE["neg"], f"ITI (n={n_valid})"),
    ]:
        ax.plot(bin_centers, mean, color=color, lw=1.8, label=label)
        ax.fill_between(bin_centers, mean - sem, mean + sem, color=color, alpha=0.25, linewidth=0)
    ax.axvline(0, color=PALETTE["neutral"], lw=0.8, ls="--", alpha=0.6)
    ax.axhline(0, color=PALETTE["neutral"], lw=0.5)
    ax.set_xlabel("Time from bout start (s)")
    ax.set_ylabel("Firing rate (z, session-wide)")
    ax.set_title("Population PETH — movement-bout starts")
    ax.legend()
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "population_overlay_bout_starts", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 6e. Waterfall

In [13]:
if ENV == "codeocean":
    OFFSET, LW, ALPHA = 0.2, 1, 0.5

    fig, axes = plt.subplots(1, 2, figsize=(11, 8), sharex=True, sharey=True)
    for ax, Z, title, color in [
        (axes[0], z_gr_sorted, "Within-trial bouts", PALETTE["pos"]),
        (axes[1], z_it_sorted, "ITI bouts",          PALETTE["neg"]),
    ]:
        n = Z.shape[0]
        for i in range(n):
            y = Z[i] + (n - 1 - i) * OFFSET
            ax.plot(bin_centers, y, color=color, lw=LW, alpha=ALPHA)
        ax.axvline(0, color=PALETTE["neutral"], lw=0.6, ls="--", alpha=0.5)
        ax.set_xlabel("Time from bout start (s)")
        ax.set_title(f"{title}  (n={n})")
    axes[0].set_ylabel(f"Unit (offset {OFFSET} z per unit, top = largest peak)")
    for ax in axes:
        ax.set_yticks([])
        style_ax(ax)
    plt.suptitle("Population PETH waterfall — movement-bout starts", y=1.00)
    plt.tight_layout()
    save_fig(fig, "population_waterfall_bout_starts", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 6f. Go-cue-aligned reference

The third condition the two bout conditions are read against: the same units'
go-cue-aligned PETH, on the same firing-rate axis (not z-scored, since the go-cue PETH
wasn't part of the z-scoring/sorting pass above).


In [14]:
if ENV == "codeocean":
    peth_cue_list = []
    for u, (sess_u, _uid) in zip(units_with_spikes.itertuples(index=False), unit_labels):
        data = session_data_cache[sess_u]
        evnts = data["events"]
        offset = evnts.loc[evnts["event"] == "goCue_start_time", "raw_timestamps"].iloc[0]
        spk = np.asarray(u.spike_times, dtype=float) - offset

        cue_times = data["trials"]["goCue_start_time_in_session"].dropna().to_numpy()
        rp_cue, _ = raster_for_event_times(spk, cue_times, PRE_POP, POST_POP, BIN_SIZE_POP)
        psth_cue, _ = compute_psth(rp_cue.raster, bin_size=rp_cue.bin_size)
        peth_cue_list.append(psth_cue)

    peth_cue = np.array(peth_cue_list)
    cue_sorted = peth_cue[valid][order]
    gr_hz_sorted = peth_gr[valid][order]
    it_hz_sorted = peth_it[valid][order]

    cue_mean, cue_sem = mean_sem(cue_sorted)
    gr_hz_mean, gr_hz_sem = mean_sem(gr_hz_sorted)
    it_hz_mean, it_hz_sem = mean_sem(it_hz_sorted)

    fig, ax = plt.subplots(figsize=(6.5, 4))
    for mean, sem, color, label in [
        (cue_mean, cue_sem, OKABE_ITO["black"], f"Go cue (n={n_valid})"),
        (gr_hz_mean, gr_hz_sem, PALETTE["pos"], f"Within-trial bout (n={n_valid})"),
        (it_hz_mean, it_hz_sem, PALETTE["neg"], f"ITI bout (n={n_valid})"),
    ]:
        ax.plot(bin_centers, mean, color=color, lw=1.8, label=label)
        ax.fill_between(bin_centers, mean - sem, mean + sem, color=color, alpha=0.2, linewidth=0)
    ax.axvline(0, color=PALETTE["neutral"], lw=0.8, ls="--", alpha=0.6)
    ax.set_xlabel("Time from event (s)")
    ax.set_ylabel("Firing rate (Hz)")
    ax.set_title("Population PETH — go cue vs movement bouts")
    ax.legend(fontsize=9)
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "population_overlay_gocue_reference", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 7. Per-unit encoding: go-responsive vs ITI Δ firing rate

**Code Ocean only — not executed this session.** Ports the paired per-unit comparison
from source cells 29–31 — the only place in the source notebook with a formal per-unit
statistical test, as opposed to a population-average figure — rebuilt on the primary
movement-derived `get_session_bout_times` (the source's cell 29 used a different, more
ad hoc event definition for this specific comparison; here it uses the same bout
definition as everything else in §5–§6, for internal consistency).


In [15]:
BASELINE_WIN_UNIT = (-0.5, 0.0)
RESPONSE_WIN_UNIT = (0.0, 0.5)


def count_spikes(spikes, event_times, window):
    """Vectorized spike count in a fixed window around each event time."""
    starts = event_times + window[0]
    ends   = event_times + window[1]
    return (np.searchsorted(spikes, ends,   side="left")
            - np.searchsorted(spikes, starts, side="left"))


def compute_unit_bout_counts(spikes, session, unit_id, gr_times, iti_times):
    pre_dur  = BASELINE_WIN_UNIT[1] - BASELINE_WIN_UNIT[0]
    post_dur = RESPONSE_WIN_UNIT[1] - RESPONSE_WIN_UNIT[0]
    spikes = np.sort(spikes)

    rows = []
    for class_label, evt in [("go_responsive", gr_times), ("ITI", iti_times)]:
        if len(evt) == 0:
            continue
        base = count_spikes(spikes, evt, BASELINE_WIN_UNIT) / pre_dur
        post = count_spikes(spikes, evt, RESPONSE_WIN_UNIT) / post_dur
        rows.append(pd.DataFrame({
            "session": session, "unit_id": unit_id, "class": class_label,
            "event_time": evt, "baseline_rate_hz": base, "spike_rate_hz": post,
            "delta_hz": post - base,
        }))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


In [16]:
if ENV == "codeocean":
    all_bout_counts = []

    for u in units_with_spikes.itertuples(index=False):
        sess_u = u.session
        gr_t, it_t = session_bout_cache[sess_u]  # already computed in §6

        evnts = session_data_cache[sess_u]["events"]
        offset = evnts.loc[evnts["event"] == "goCue_start_time", "raw_timestamps"].iloc[0]
        spk = np.asarray(u.spike_times, dtype=float) - offset

        uc = compute_unit_bout_counts(spk, sess_u, u.unit_id, gr_t, it_t)
        if len(uc):
            all_bout_counts.append(uc)

    bout_counts_df = pd.concat(all_bout_counts, ignore_index=True)

    unit_mean = (bout_counts_df.groupby(["session", "unit_id", "class"])["delta_hz"]
                 .mean().unstack("class"))
    paired = unit_mean.dropna(subset=["go_responsive", "ITI"])
    print(f"Paired units (both classes present): {len(paired)} / {len(unit_mean)}")

    gr_vals = paired["go_responsive"].to_numpy()
    it_vals = paired["ITI"].to_numpy()
    d = gr_vals - it_vals
    d_nz = d[d != 0]
    stat_p = wilcoxon(d_nz).pvalue if len(d_nz) else np.nan
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 7b. Paired go-responsive vs ITI Δ firing rate

In [17]:
if ENV == "codeocean":
    fig, ax = plt.subplots(figsize=(3.2, 4.2))
    x_pos = np.array([0.0, 1.0])
    for g, i in zip(gr_vals, it_vals):
        ax.plot(x_pos, [g, i], "-", color=PALETTE["neutral"], alpha=0.1, lw=1)
    ax.scatter(np.zeros_like(gr_vals), gr_vals, s=15, alpha=0.4, color=PALETTE["pos"])
    ax.scatter(np.ones_like(it_vals),  it_vals, s=15, alpha=0.4, color=PALETTE["neg"])

    means = [gr_vals.mean(), it_vals.mean()]
    sems  = [gr_vals.std(ddof=1) / np.sqrt(len(gr_vals)), it_vals.std(ddof=1) / np.sqrt(len(it_vals))]
    ax.errorbar(x_pos, means, sems, fmt="o", color=OKABE_ITO["black"], capsize=4, lw=2, zorder=5)
    ax.plot(x_pos, means, "-", color=OKABE_ITO["black"], lw=2)

    ax.axhline(0, color=PALETTE["neutral"], lw=0.5, ls="--")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(["Go-responsive", "ITI bouts"])
    ax.set_ylabel("Δ firing rate (Hz, post - baseline)")
    ax.set_title(f"n = {len(paired)} units\nWilcoxon p = {stat_p:.2g}")
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "per_unit_paired_delta_hz", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 7c. Baseline vs response, per class, with per-unit significance

In [18]:
if ENV == "codeocean":
    unit_rates = (bout_counts_df.groupby(["session", "unit_id", "class"])
                  [["baseline_rate_hz", "spike_rate_hz"]].mean().reset_index())

    def summarize_unit(g):
        pre, post = g["baseline_rate_hz"].to_numpy(), g["spike_rate_hz"].to_numpy()
        dd = post - pre
        dd = dd[np.isfinite(dd) & (dd != 0)]
        p = wilcoxon(dd, alternative="two-sided").pvalue if len(dd) else np.nan
        return pd.Series({"p": p})

    unit_stats = (bout_counts_df.groupby(["session", "unit_id", "class"], sort=False)
                  .apply(summarize_unit).reset_index())
    unit_rates = unit_rates.merge(unit_stats, on=["session", "unit_id", "class"], how="left")
    unit_rates["sig"] = unit_rates["p"] < 0.05

    fig, axes = plt.subplots(1, 2, figsize=(6.4, 4.2), sharey=True)
    for ax, cls in [(axes[0], "go_responsive"), (axes[1], "ITI")]:
        sub = unit_rates[unit_rates["class"] == cls]
        y_pre, y_post, sig = sub["baseline_rate_hz"].to_numpy(), sub["spike_rate_hz"].to_numpy(), sub["sig"].to_numpy()

        dd = y_post - y_pre
        dd_nz = dd[dd != 0]
        p_group = wilcoxon(dd_nz).pvalue if len(dd_nz) else np.nan
        n_sig, n_tot = int(sig.sum()), int(np.isfinite(sub["p"]).sum())

        x_pos = np.array([0.0, 1.0])
        for yp, yo, s in zip(y_pre, y_post, sig):
            c = PALETTE["sig"] if s else PALETTE["not_sig"]
            ax.plot(x_pos, [yp, yo], "-", color=c, alpha=(0.5 if s else 0.25), lw=1,
                    zorder=(2 if s else 1))

        ns = ~sig
        ax.scatter(np.zeros(ns.sum()), y_pre[ns],  s=15, alpha=0.5, color=PALETTE["not_sig"], zorder=1)
        ax.scatter(np.ones(ns.sum()),  y_post[ns], s=15, alpha=0.5, color=PALETTE["not_sig"], zorder=1)
        ax.scatter(np.zeros(sig.sum()), y_pre[sig],  s=18, alpha=0.7, color=PALETTE["sig"], zorder=3)
        ax.scatter(np.ones(sig.sum()),  y_post[sig], s=18, alpha=0.7, color=PALETTE["sig"], zorder=3)

        means = [y_pre.mean(), y_post.mean()]
        sems  = [y_pre.std(ddof=1) / np.sqrt(len(y_pre)), y_post.std(ddof=1) / np.sqrt(len(y_post))]
        ax.errorbar(x_pos, means, sems, fmt="o", color=OKABE_ITO["black"], capsize=4, lw=2, zorder=5)
        ax.plot(x_pos, means, "-", color=OKABE_ITO["black"], lw=2, zorder=5)

        ax.set_xticks(x_pos)
        ax.set_xticklabels(["Baseline", "Response"])
        ax.set_title(f"{cls}\nn = {len(sub)}, p = {p_group:.2g}\nSignificant units: {n_sig}/{n_tot}")
        style_ax(ax)

    axes[0].set_ylabel("Firing rate (Hz)")
    plt.tight_layout()
    save_fig(fig, "per_unit_baseline_vs_response", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 7d. Go-responsive vs ITI Δ firing rate, per unit

In [19]:
if ENV == "codeocean":
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(gr_vals, it_vals, s=20, alpha=0.5, color=OKABE_ITO["black"], edgecolor="none")

    lo, hi = min(gr_vals.min(), it_vals.min()), max(gr_vals.max(), it_vals.max())
    pad = 0.05 * (hi - lo)
    lims = (lo - pad, hi + pad)
    ax.plot(lims, lims, "--", color=PALETTE["neutral"], lw=0.8, alpha=0.5)
    ax.axhline(0, color=PALETTE["neutral"], lw=0.5, ls=":")
    ax.axvline(0, color=PALETTE["neutral"], lw=0.5, ls=":")
    ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect("equal")
    ax.set_xlabel("Δ firing rate — go-responsive (Hz)")
    ax.set_ylabel("Δ firing rate — ITI bouts (Hz)")
    ax.set_title(f"n = {len(paired)} units\nWilcoxon p = {stat_p:.2g}")
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "per_unit_gr_vs_iti_scatter", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 7e. Register through `per_unit_stats_registry` / `encoding_methods`

Reframes the per-unit go-responsive-vs-ITI comparison as an `AnalysisSpec` /
`fit_encoding` call — response = `delta_hz`, predictor = a binary `is_iti` dummy,
method = OLS — so the per-unit T-statistic is the standard schema `PerUnitStatsRegistry`
expects, and this result composes with `eph_01`–`eph_04` (comparable via
`reg.compare()` / `reg.screen()` in a downstream notebook) rather than living only as
a one-off Wilcoxon test.


In [20]:
if ENV == "codeocean":
    from encoding_methods import AnalysisSpec, fit_encoding
    from per_unit_stats_registry import PerUnitStatsRegistry
    from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix

    event_df = bout_counts_df.copy()
    event_df["is_iti"] = (event_df["class"] == "ITI").astype(float)

    bout_class_spec = AnalysisSpec(
        name="eph07_bout_class_delta_hz",
        predictor_col="is_iti",
        response_col="delta_hz",
        method="ols",
        zscore_x=False,
        min_trials=5,
        notes="Per-unit go-responsive vs ITI bout delta_hz, movement-derived bouts "
              f"(gap={GAP_THRESHOLD_S}s, go_resp_window={GO_RESPONSE_WINDOW_S}s, "
              f"iti_post_cue>{ITI_MIN_POST_CUE_S}s, iti_pre_next>{ITI_MIN_PRE_NEXT_S}s). "
              "Positive T = higher delta_hz for ITI bouts than go-responsive bouts.",
    )
    bout_class_result = fit_encoding(event_df, bout_class_spec)

    reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)
    reg.register(bout_class_result)
    print(reg)
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 8. Qualitative example: event raster + single-trial timeseries

**Code Ocean only — not executed this session.** The example-session event raster and
single-trial tongue-position + event overlay from source cells 35–36, restyled with
`PALETTE`. Cell 37 (video-clip extraction around specific bouts) is dropped — a
one-off exploratory aside, and the clip-extraction helpers are library-owned
(`aind_dynamic_foraging_behavior_video_analysis...video_clip_utils`), not something
this notebook should re-invoke.


In [21]:
if ENV == "codeocean":
    data = session_data_cache[session]
    movs_s, licks_s, trials_s = data["movs"], data["licks"], data["trials"]

    WIN = (-4.0, 4.0)
    cue_times_series = trials_s.set_index("trial")["goCue_start_time_in_session"].dropna()
    cue_times_arr = np.sort(cue_times_series.to_numpy())
    mov_starts  = np.sort(movs_s["start_time"].dropna().to_numpy())
    lick_times  = np.sort(licks_s["timestamps"].dropna().to_numpy())
    bout_starts = np.sort(licks_s.loc[licks_s["bout_start"] == True, "timestamps"].to_numpy())

    fig, ax = plt.subplots(figsize=(6, 8))
    for trial_id, t0 in cue_times_series.items():
        m = mov_starts[(mov_starts >= t0 + WIN[0]) & (mov_starts <= t0 + WIN[1])] - t0
        l = lick_times[(lick_times >= t0 + WIN[0]) & (lick_times <= t0 + WIN[1])] - t0
        b = bout_starts[(bout_starts >= t0 + WIN[0]) & (bout_starts <= t0 + WIN[1])] - t0
        c = cue_times_arr[(cue_times_arr >= t0 + WIN[0]) & (cue_times_arr <= t0 + WIN[1])] - t0
        c = c[c != 0]

        if len(m):
            ax.scatter(m, np.full_like(m, trial_id), s=6, color=PALETTE["neg"], alpha=0.6, marker="|", linewidths=0.8)
        if len(l):
            ax.scatter(l, np.full_like(l, trial_id), s=10, color=PALETTE["baseline"], alpha=0.6, marker="|", linewidths=0.9)
        if len(b):
            ax.scatter(b, np.full_like(b, trial_id), s=32, color=PALETTE["accent"], alpha=0.9, marker="|", linewidths=1.3)
        if len(c):
            ax.scatter(c, np.full_like(c, trial_id), s=32, color=OKABE_ITO["black"], alpha=0.8, marker="|", linewidths=1.3)

    ax.axvline(0, color=OKABE_ITO["black"], lw=0.8, ls="--", alpha=0.6)
    ax.set_xlim(WIN)
    ax.set_ylim(cue_times_series.index.max(), cue_times_series.index.min())
    ax.set_xlabel("Time from go cue (s)")
    ax.set_ylabel("Trial")
    ax.set_title(f"{session}\nblue = tongue movement, green = lick, purple = lick-bout start, black = neighboring go cue")
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "example_session_event_raster", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


In [22]:
if ENV == "codeocean":
    from matplotlib.lines import Line2D

    TRIAL_ID = int(trials_s["trial"].dropna().median())
    WIN = (-4.0, 4.0)
    kins_s = data["kins"]

    t0 = trials_s.set_index("trial").loc[TRIAL_ID, "goCue_start_time_in_session"]
    t_lo, t_hi = t0 + WIN[0], t0 + WIN[1]

    k = kins_s[(kins_s["time_in_session"] >= t_lo) & (kins_s["time_in_session"] <= t_hi)]
    k_t, k_x = k["time_in_session"].to_numpy() - t0, k["x"].to_numpy()

    def in_win(arr):
        arr = np.asarray(arr)
        return arr[(arr >= t_lo) & (arr <= t_hi)] - t0

    m = in_win(movs_s["start_time"].dropna().to_numpy())
    l = in_win(licks_s["timestamps"].dropna().to_numpy())
    b = in_win(licks_s.loc[licks_s["bout_start"] == True, "timestamps"].to_numpy())
    c = in_win(trials_s["goCue_start_time_in_session"].dropna().to_numpy())
    c = c[c != 0]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(k_t, k_x, s=6, color=PALETTE["neutral"], alpha=0.7, label="tongue x")

    ymin, ymax = ax.get_ylim()
    span = ymax - ymin
    ax.set_ylim(ymin - 0.1 * span, ymax + 0.25 * span)

    for x in m: ax.axvline(x, color=PALETTE["neg"], alpha=0.5, lw=1)
    for x in l: ax.axvline(x, color=PALETTE["baseline"], alpha=0.4, lw=1)
    for x in b: ax.axvline(x, color=PALETTE["accent"], alpha=0.8, lw=1.5)
    for x in c: ax.axvline(x, color=OKABE_ITO["black"], alpha=0.7, lw=1.2, ls="--")
    ax.axvline(0, color=OKABE_ITO["black"], lw=1.5, ls="-", alpha=0.9)

    legend_elems = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=PALETTE["neutral"], markersize=5, label="tongue x"),
        Line2D([0], [0], color=OKABE_ITO["black"], lw=1.5, label="go cue (this trial)"),
        Line2D([0], [0], color=OKABE_ITO["black"], lw=1.2, ls="--", label="go cue (neighbor)"),
        Line2D([0], [0], color=PALETTE["neg"], lw=1, label="movement start"),
        Line2D([0], [0], color=PALETTE["baseline"], lw=1, label="lick"),
        Line2D([0], [0], color=PALETTE["accent"], lw=1.5, label="lick-bout start"),
    ]
    ax.legend(handles=legend_elems, loc="upper right", fontsize=8)
    ax.set_xlim(WIN)
    ax.set_xlabel("Time from go cue (s)")
    ax.set_ylabel("Tongue x position")
    ax.set_title(f"{session} — trial {TRIAL_ID}")
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "example_trial_timeseries", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 9. Robustness check: lick-bout-derived definition

**Code Ocean only — not executed this session.** Repeats the go-responsive-vs-ITI
Δ firing rate comparison (§7) using `licks["bout_start"]` as the bout onset instead of
movement bouts — the lick-derived definition from source cells 26–32.

One deliberate departure from the source: cell 26's own go-responsive class wasn't
lick-bout-derived at all — it used first-in-trial movement latency (`movs`), while only
the ITI class came from `licks["bout_start"]`, mixing the two definitions within what
was billed as the "lick bout" analysis. For a clean apples-to-apples robustness check,
this section classifies lick-bout starts the same way §5–§7 classify movement-bout
starts — both go-responsive and ITI computed from `licks["bout_start"]` via the shared
`classify_bout_times` — using the thresholds from source cell 26's own header
(`GO_RESPONSE_WINDOW_S=2.0, ITI_MIN_POST_CUE_S=1.5, ITI_MIN_PRE_NEXT_S=1.0`), distinct
from §3's movement-derived primary set. Reported at lower weight than §7: a summary
statistic and a registry comparison against the primary result, not the full plot
suite.


In [23]:
GO_RESPONSE_WINDOW_S_LICK = 2.0
ITI_MIN_POST_CUE_S_LICK   = 1.5
ITI_MIN_PRE_NEXT_S_LICK   = 1.0


def get_session_bout_times_lick(licks, trials, go_response_window_s, iti_min_post_cue_s, iti_min_pre_next_s):
    """Lick-bout-derived within-trial vs ITI event times for one session."""
    bout_starts = licks.loc[licks["bout_start"] == True, "timestamps"].dropna().to_numpy()
    go_cues = trials["goCue_start_time_in_session"].dropna().to_numpy()
    return classify_bout_times(
        bout_starts, go_cues,
        go_response_window_s=go_response_window_s,
        iti_min_post_cue_s=iti_min_post_cue_s,
        iti_min_pre_next_s=iti_min_pre_next_s,
    )


In [24]:
if ENV == "codeocean":
    all_bout_counts_lick = []

    for u in units_with_spikes.itertuples(index=False):
        sess_u = u.session
        data = session_data_cache[sess_u]

        gr_t_lick, it_t_lick = get_session_bout_times_lick(
            data["licks"], data["trials"],
            go_response_window_s=GO_RESPONSE_WINDOW_S_LICK,
            iti_min_post_cue_s=ITI_MIN_POST_CUE_S_LICK,
            iti_min_pre_next_s=ITI_MIN_PRE_NEXT_S_LICK,
        )

        evnts = data["events"]
        offset = evnts.loc[evnts["event"] == "goCue_start_time", "raw_timestamps"].iloc[0]
        spk = np.asarray(u.spike_times, dtype=float) - offset

        uc = compute_unit_bout_counts(spk, sess_u, u.unit_id, gr_t_lick, it_t_lick)
        if len(uc):
            all_bout_counts_lick.append(uc)

    bout_counts_df_lick = pd.concat(all_bout_counts_lick, ignore_index=True)

    unit_mean_lick = (bout_counts_df_lick.groupby(["session", "unit_id", "class"])["delta_hz"]
                       .mean().unstack("class"))
    paired_lick = unit_mean_lick.dropna(subset=["go_responsive", "ITI"])
    d_lick = paired_lick["go_responsive"].to_numpy() - paired_lick["ITI"].to_numpy()
    d_lick_nz = d_lick[d_lick != 0]
    stat_p_lick = wilcoxon(d_lick_nz).pvalue if len(d_lick_nz) else np.nan

    print(f"Lick-bout-derived: {len(paired_lick)} paired units, "
          f"mean delta (go_resp - ITI) = {d_lick.mean():.3f} Hz, Wilcoxon p = {stat_p_lick:.3g}")
    print(f"Movement-bout-derived (§7, primary): {len(paired)} paired units, "
          f"mean delta (go_resp - ITI) = {d.mean():.3f} Hz, Wilcoxon p = {stat_p:.3g}")

    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(paired_lick["go_responsive"], paired_lick["ITI"], s=20, alpha=0.5,
               color=PALETTE["accent"], edgecolor="none")
    lo = min(paired_lick["go_responsive"].min(), paired_lick["ITI"].min())
    hi = max(paired_lick["go_responsive"].max(), paired_lick["ITI"].max())
    pad = 0.05 * (hi - lo)
    ax.plot((lo - pad, hi + pad), (lo - pad, hi + pad), "--", color=PALETTE["neutral"], lw=0.8, alpha=0.5)
    ax.set_xlabel("Δ firing rate — go-responsive, lick-bout (Hz)")
    ax.set_ylabel("Δ firing rate — ITI, lick-bout (Hz)")
    ax.set_title(f"Robustness check (lick-derived)\nn = {len(paired_lick)}, Wilcoxon p = {stat_p_lick:.2g}")
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "robustness_lick_bout_scatter", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    # ---- register and compare against the movement-derived primary ----
    event_df_lick = bout_counts_df_lick.copy()
    event_df_lick["is_iti"] = (event_df_lick["class"] == "ITI").astype(float)
    bout_class_spec_lick = AnalysisSpec(
        name="eph07_bout_class_delta_hz_lick",
        predictor_col="is_iti", response_col="delta_hz", method="ols",
        zscore_x=False, min_trials=5,
        notes="Robustness check: same comparison as eph07_bout_class_delta_hz, "
              "lick-bout-derived instead of movement-bout-derived.",
    )
    bout_class_result_lick = fit_encoding(event_df_lick, bout_class_spec_lick)
    reg.register(bout_class_result_lick)

    cmp = reg.compare("eph07_bout_class_delta_hz", "eph07_bout_class_delta_hz_lick")
    rho, p_rho = spearmanr(cmp["t_a"], cmp["t_b"])
    print(f"\nAgreement between movement- and lick-derived per-unit T-stats: "
          f"n={len(cmp)}, Spearman rho={rho:.3f} (p={p_rho:.3g})")
    print(cmp["sig_category"].value_counts())
else:
    print("ENV == 'local': skipping — needs per-session spike times.")


ENV == 'local': skipping — needs per-session spike times.


## 10. Summary

- **§4 (bout helpers) ran locally and is verified**: `annotate_movement_bouts` and
  `get_session_bout_times` behave as expected against the pooled parquet — bout counts,
  size distribution, and a plausible within-trial/ITI split per session, all printed
  above.
- **Everything from §5 onward is written but unexecuted.** It needs per-session
  intermediate data and spike times that only exist on Code Ocean. Nothing about the
  population PETH shape, the per-unit Wilcoxon result, or the movement-vs-lick-bout
  agreement in §9 should be treated as a finding until it's actually run there.
- Once run and reviewed, this notebook is the sole remaining gate on archiving
  `tongue_kinematics_ephys_intertrialmovs.ipynb` (see `TODO.md` / `REORG.md`) — but
  archiving itself is a separate step, after review.
